# Свой голос для TeraTTS

Блокнот подбирает **файл голоса** (JSON, как `ru_f1`/`st3_M1`) по вашей записи. Сама модель не обучается: подбираются только числа, описывающие тембр и темп речи, — прямо через TeraTTS, на русских фразах.

Инструмент: [supertonic.embed](https://github.com/kdrkdrkdr/supertonic.embed) (MIT), подключён к TeraTTS v2.

**Перед запуском:** меню *Среда выполнения → Сменить среду выполнения → T4 GPU*. Потом запускайте ячейки по порядку (▶ слева от каждой).

**Запись:** 20–60 секунд спокойного чтения, тихая комната, без музыки и длинных пауз. Подойдёт m4a/mp3/wav с диктофона телефона. Записывать можно только свой голос или голос человека, который согласен.

In [ ]:
# 1. Видеокарта на месте?
import torch, subprocess
assert torch.cuda.is_available(), "Нет GPU: Среда выполнения → Сменить среду выполнения → T4 GPU"
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout)

In [ ]:
# 2. Инструмент и зависимости (~1–2 мин)
%cd /content
!rm -rf supertonic.embed && git clone -q https://github.com/kdrkdrkdr/supertonic.embed.git
!git -C supertonic.embed log -1 --format="supertonic.embed %h %cd"
!pip -q install onnx onnxruntime onnx2torch onnxslim librosa pyyaml

In [ ]:
# 3. Модели TeraTTS (зафиксированная версия) и голоса-заготовки (~400 МБ)
import os, json, shutil, numpy as np
from huggingface_hub import hf_hub_download
REV = "f05ea799094571a3553904a555df3834fb0b963b"
TERA = "/content/tera"; MODELS = f"{TERA}/models"; PRESETS = f"{TERA}/presets"
os.makedirs(MODELS, exist_ok=True); os.makedirs(PRESETS, exist_ok=True)
get = lambda f: hf_hub_download("TeraSpace/TeraTTSv2", f, revision=REV)
for f in ["text_encoder.onnx", "duration_predictor.onnx", "sampler_distilled_cfg3_8step.onnx", "vocoder.onnx"]:
    shutil.copy(get(f"models/{f}"), f"{MODELS}/{f}")
shutil.copy(get("unicode_indexer.json"), f"{MODELS}/unicode_indexer.json")

def style_json(ttl, dp):
    return {"style_ttl": {"data": ttl.tolist(), "dims": [1, 50, 256], "type": "float32"},
            "style_dp": {"data": dp.tolist(), "dims": [1, 8, 16], "type": "float32"}}
# голоса TeraTTS (имена на F*/M*, как ждёт инструмент)
for v, name in [("ru_f1", "F_ru1"), ("ru_f2", "F_ru2"), ("ru_m1", "M_ru1"), ("ru_m5", "M_ru5")]:
    ttl = np.load(get(f"styles/{v}/style_ttl.npy")).astype(np.float32)
    dp = np.load(get(f"styles/{v}/style_dp.npy")).astype(np.float32)
    json.dump(style_json(ttl, dp), open(f"{PRESETS}/{name}.json", "w"))
# голоса Supertonic 3 — подходят TeraTTS (проверено на слух)
for v in ["M1", "M2", "M3", "M4", "M5", "F1", "F2", "F3", "F4", "F5"]:
    shutil.copy(hf_hub_download("Supertone/supertonic-3", f"voice_styles/{v}.json"), f"{PRESETS}/{v}.json")
print("Готово:", sorted(os.listdir(MODELS)), len(os.listdir(PRESETS)), "заготовок")

In [ ]:
# 4. Загрузите запись голоса (кнопка «Выбрать файлы» появится ниже)
from google.colab import files
import librosa
up = files.upload()
src = next(iter(up))
!ffmpeg -y -loglevel error -i "{src}" -ac 1 -ar 44100 /content/my_voice.wav
y, sr = librosa.load("/content/my_voice.wav", sr=None)
speech = librosa.effects.split(y, top_db=35)
voiced = sum(e - s for s, e in speech) / sr
print(f"Длина {len(y)/sr:.1f} с, из них речь ≈ {voiced:.1f} с")
if voiced < 10: print("⚠️ Маловато речи — лучше 20–60 секунд.")
open("/content/list.txt", "w").write("/content/my_voice.wav|my_voice\n")

In [ ]:
# 5. Подключаем TeraTTS к инструменту и проверяем конвертацию моделей
import yaml, sys, subprocess, textwrap
REPO = "/content/supertonic.embed"; SRC = f"{REPO}/src"
cfg = yaml.safe_load(open(f"{SRC}/config.yaml"))
PROMPTS_RU = [
    "<ru>С+олнце сад+ится з+а г+оры, окр+ашивая н+ебо в р+озовые и ор+анжевые тон+а.</ru>",
    "<ru>Технол+огии измен+или т+о, как л+юди +общаются др+уг с др+угом по всем+у м+иру.</ru>",
    "<ru>Он+а шл+а по т+ихому л+есу, сл+ушая, как по+ют пт+ицы в ветв+ях дер+евьев.</ru>",
    "<ru>На+учные иссл+едования тр+ебуют терп+ения, вним+ательности и тщ+ательного ан+ализа.</ru>",
    "<ru>В стар+инной библиот+еке хран+ились т+ысячи кн+иг, с+обранных за мн+ого покол+ений.</ru>",
]
if not os.path.exists(f"{SRC}/frozen_tts_upstream.py"):
    shutil.copy(f"{SRC}/frozen_tts.py", f"{SRC}/frozen_tts_upstream.py")
FROZEN_TERA = "\"\"\"TeraTTS v2 as frozen, differentiable PyTorch — drop-in for supertonic.embed's frozen_tts.py.\n\nSame public surface (FrozenTTS, Prompt, build_prompts), TeraTTS pipeline inside:\n  * text encoder gets the text WITH '+' stress marks, the duration predictor WITHOUT;\n  * one call of the distilled sampler (8 Euler steps baked in) instead of a loop;\n  * latent: 144 channels, 3072 samples per frame; duration / 1.05.\nEvery converted module is checked against ONNX Runtime before use.\n\"\"\"\nimport json\nimport os\nimport unicodedata\nfrom dataclasses import dataclass\n\nimport numpy as np\nimport onnx\nimport onnxruntime as ort\nimport onnxslim\nimport torch\nfrom onnx2torch import convert\n\nfrom config import DEVICE, ONNX_DIR, SEED, SPEED, SR, TEXTS\nfrom style_file import preset_path, read_style\n\nSIZING_PRESET = \"M1\"\nGUIDANCE = 3.0\nSAMPLER_FILE = \"sampler_distilled_cfg3_8step.onnx\"\nSAMPLES_PER_FRAME = 3072\nLATENT_DIM = 144\n\n\ndef _convert(path):\n    \"\"\"Slim and convert one ONNX module; retry with opset forced to 17 (as upstream).\"\"\"\n    errors = []\n    for force17 in (False, True):\n        try:\n            model = onnxslim.slim(path)\n            if force17:\n                for opset in model.opset_import:\n                    if opset.domain in (\"\", \"ai.onnx\"):\n                        opset.version = 17\n            for node in model.graph.node:\n                if node.op_type == \"Clip\":\n                    kept = [i for i in node.input if i != \"\"]\n                    del node.input[:]\n                    node.input.extend(kept)\n            module = convert(model).eval()\n            for p in module.parameters():\n                p.requires_grad_(False)\n            return module.to(DEVICE)\n        except Exception as e:  # noqa: BLE001\n            errors.append(f\"{'opset17' if force17 else 'as-is'}: {e}\")\n    raise RuntimeError(f\"cannot convert {os.path.basename(path)}: \" + \" | \".join(errors))\n\n\ndef _verify(module, path, feeds):\n    \"\"\"Converted module must reproduce ONNX Runtime on the same inputs.\"\"\"\n    sess = ort.InferenceSession(path, providers=[\"CPUExecutionProvider\"])\n    want = sess.run(None, feeds)[0]\n    with torch.no_grad():\n        got = module(*[torch.tensor(v, device=DEVICE) for v in feeds.values()])\n    got = (got[0] if isinstance(got, (tuple, list)) else got).cpu().numpy()\n    err = float(np.max(np.abs(got.reshape(want.shape) - want)))\n    scale = float(np.max(np.abs(want))) + 1e-6\n    print(f\"   {os.path.basename(path):<40} max diff {err:.2e} (scale {scale:.2f})\")\n    if err > 1e-2 * max(scale, 1.0):\n        raise RuntimeError(f\"{os.path.basename(path)}: converted module differs from ONNX ({err})\")\n\n\nclass _NoStress(torch.nn.Module):\n    \"\"\"Duration predictor that drops '+' tokens: TeraTTS times the unmarked text.\"\"\"\n\n    def __init__(self, dp, plus_id):\n        super().__init__()\n        self.dp = dp\n        self.plus_id = plus_id\n\n    def forward(self, ids, style_dp, mask):\n        keep = ids[0] != self.plus_id\n        return self.dp(ids[:, keep], style_dp, mask[..., keep]).reshape(-1)\n\n\nclass TeraTokenizer:\n    def __init__(self, indexer_path):\n        with open(indexer_path, encoding=\"utf-8\") as f:\n            self.table = json.load(f)\n\n    def __call__(self, text, lang=\"ru\"):\n        if not text.lstrip().startswith(\"<\"):\n            text = f\"<{lang}>{text}</{lang}>\"\n        text = unicodedata.normalize(\"NFKD\", text)\n        ids = np.array([[self.table[ord(c)] for c in text]], dtype=np.int64)\n        return ids, np.ones((1, 1, ids.shape[1]), dtype=np.float32)\n\n\n@dataclass\nclass Prompt:\n    ids: torch.Tensor\n    mask: torch.Tensor\n    latent: torch.Tensor\n    latent_mask: torch.Tensor\n\n    def repeat(self, n):\n        return (self.ids.expand(n, -1).contiguous(),\n                self.mask.expand(n, -1, -1).contiguous(),\n                self.latent.expand(n, -1, -1).contiguous(),\n                self.latent_mask.expand(n, -1, -1).contiguous())\n\n\nclass FrozenTTS:\n    def __init__(self):\n        root = os.path.dirname(os.path.abspath(ONNX_DIR.rstrip(\"/\")))\n        indexer = os.path.join(ONNX_DIR, \"unicode_indexer.json\")\n        if not os.path.exists(indexer):\n            indexer = os.path.join(root, \"unicode_indexer.json\")\n        self.tokenize = TeraTokenizer(indexer)\n        self.chunk = SAMPLES_PER_FRAME\n        self.latent_dim = LATENT_DIM\n        p = lambda f: os.path.join(ONNX_DIR, f)\n        print(\"Converting and verifying TeraTTS modules:\")\n        rng = np.random.default_rng(0)\n        ids, mask = self.tokenize(\"<ru>Пров+ерка конверт+ации.</ru>\")\n        ttl, dp = (t.cpu().numpy() for t in read_style(preset_path(SIZING_PRESET)))\n\n        enc = _convert(p(\"text_encoder.onnx\"))\n        _verify(enc, p(\"text_encoder.onnx\"), {\"text_ids\": ids, \"style_ttl\": ttl, \"text_mask\": mask})\n        dur = _convert(p(\"duration_predictor.onnx\"))\n        plus = self.tokenize.table[ord(\"+\")]\n        dids = ids[:, ids[0] != plus]\n        _verify(dur, p(\"duration_predictor.onnx\"), {\"text_ids\": dids, \"style_dp\": dp,\n                                                   \"text_mask\": np.ones((1, 1, dids.shape[1]), np.float32)})\n        sampler = _convert(p(SAMPLER_FILE))\n        emb = ort.InferenceSession(p(\"text_encoder.onnx\"), providers=[\"CPUExecutionProvider\"]).run(\n            None, {\"text_ids\": ids, \"style_ttl\": ttl, \"text_mask\": mask})[0]\n        n = 20\n        _verify(sampler, p(SAMPLER_FILE), {\n            \"initial_latent\": rng.standard_normal((1, LATENT_DIM, n)).astype(np.float32),\n            \"text_emb\": emb, \"style_ttl\": ttl,\n            \"latent_mask\": np.ones((1, 1, n), np.float32), \"text_mask\": mask,\n            \"guidance\": np.asarray([GUIDANCE], np.float32)})\n        voc = _convert(p(\"vocoder.onnx\"))\n        _verify(voc, p(\"vocoder.onnx\"), {\"latent\": rng.standard_normal((1, LATENT_DIM, n)).astype(np.float32)})\n\n        self.text_encoder, self.sampler, self.vocoder = enc, sampler, voc\n        self.duration = _NoStress(dur, plus)\n\n    def encode(self, text, lang=\"ru\"):\n        ids, mask = self.tokenize(text, lang)\n        return (torch.tensor(ids, dtype=torch.int64, device=DEVICE),\n                torch.tensor(mask, dtype=torch.float32, device=DEVICE))\n\n    def forward(self, ids, mask, style_ttl, style_dp, latent, latent_mask):\n        b = style_ttl.shape[0]\n        seconds = self.duration(ids, style_dp, mask) / SPEED\n        text_emb = self.text_encoder(ids, style_ttl, mask)\n        g = torch.full((b,), GUIDANCE, device=DEVICE)\n        latent = self.sampler(latent * latent_mask, text_emb, style_ttl, latent_mask, mask, g)\n        if isinstance(latent, (tuple, list)):\n            latent = latent[0]\n        wav = self.vocoder(latent)\n        if isinstance(wav, (tuple, list)):\n            wav = wav[0]\n        return wav, seconds\n\n    def predict_seconds(self, prompt, style_dp):\n        with torch.no_grad():\n            return float((self.duration(prompt.ids, style_dp, prompt.mask) / SPEED).reshape(()))\n\n    def synthesize(self, prompt, style_ttl, style_dp):\n        with torch.no_grad():\n            wav, _ = self.forward(prompt.ids, prompt.mask, style_ttl, style_dp,\n                                  prompt.latent, prompt.latent_mask)\n        return wav.reshape(-1)\n\n    def synthesize_at_own_length(self, prompt, style_ttl, style_dp):\n        seconds = self.predict_seconds(prompt, style_dp)\n        frames = max(4, int(np.ceil(int(seconds * SR) / self.chunk)))\n        generator = torch.Generator(device=\"cpu\").manual_seed(SEED)\n        latent = torch.randn(1, self.latent_dim, frames, generator=generator).to(DEVICE)\n        latent_mask = torch.ones(1, 1, frames, device=DEVICE)\n        with torch.no_grad():\n            wav, _ = self.forward(prompt.ids, prompt.mask, style_ttl, style_dp, latent, latent_mask)\n        return wav.reshape(-1).cpu().numpy(), seconds\n\n\ndef build_prompts(tts, texts=TEXTS):\n    torch.manual_seed(SEED)\n    np.random.seed(SEED)\n    _, sizing_dp = read_style(preset_path(SIZING_PRESET))\n    prompts = []\n    for text in texts:\n        ids, mask = tts.encode(text)\n        with torch.no_grad():\n            seconds = float((tts.duration(ids, sizing_dp, mask) / SPEED).reshape(()))\n        frames = int(np.ceil(int(seconds * SR) / tts.chunk))\n        prompts.append(Prompt(\n            ids=ids, mask=mask,\n            latent=torch.tensor(np.random.randn(1, tts.latent_dim, frames).astype(np.float32), device=DEVICE),\n            latent_mask=torch.ones(1, 1, frames, device=DEVICE)))\n    return prompts\n"
def use_teratts():
    cfg["model"]["onnx_dir"] = MODELS; cfg["model"]["presets"] = PRESETS
    cfg["prompts"] = PROMPTS_RU
    yaml.safe_dump(cfg, open(f"{SRC}/config.yaml", "w"), allow_unicode=True)
    open(f"{SRC}/frozen_tts.py", "w").write(FROZEN_TERA)
    return subprocess.run([sys.executable, "-c", "from frozen_tts import FrozenTTS; FrozenTTS(); print('OK')"],
                          cwd=SRC, capture_output=True, text=True)

def use_supertonic3():
    st3 = "/content/st3/onnx"; os.makedirs(st3, exist_ok=True)
    for f in ["duration_predictor.onnx", "text_encoder.onnx", "vector_estimator.onnx", "vocoder.onnx", "tts.json", "unicode_indexer.json"]:
        shutil.copy(hf_hub_download("Supertone/supertonic-3", f"onnx/{f}"), f"{st3}/{f}")
    shutil.copy(f"{SRC}/frozen_tts_upstream.py", f"{SRC}/frozen_tts.py")
    t = open(f"{SRC}/text_tokenizer.py").read()
    if '"ru"' not in t:
        t = t.replace('LANGS = ["en", "ko", "es", "pt", "fr"]', 'LANGS = ["en", "ko", "es", "pt", "fr", "ru"]')
        t = t.replace('def __call__(self, text, lang="en"):', 'def __call__(self, text, lang="ru"):')
        open(f"{SRC}/text_tokenizer.py", "w").write(t)
    f = open(f"{SRC}/frozen_tts.py").read().replace('def encode(self, text, lang="en"):', 'def encode(self, text, lang="ru"):')
    open(f"{SRC}/frozen_tts.py", "w").write(f)
    cfg["model"]["onnx_dir"] = st3; cfg["model"]["presets"] = PRESETS
    cfg["prompts"] = [p.replace("<ru>", "").replace("</ru>", "").replace("+", "") for p in PROMPTS_RU]
    yaml.safe_dump(cfg, open(f"{SRC}/config.yaml", "w"), allow_unicode=True)

r = use_teratts()
print(r.stdout[-3000:])
if "OK" in r.stdout:
    ENGINE = "teratts"; print("✅ Голос подбирается прямо через TeraTTS.")
else:
    print(r.stderr[-2500:])
    print("⚠️ TeraTTS не конвертировалась — переключаюсь на Supertonic 3 (её голоса подходят TeraTTS).")
    use_supertonic3(); ENGINE = "supertonic3"

In [ ]:
# 6. Подбор голоса (≈5–20 минут на T4; прогресс печатается ниже)
%cd /content/supertonic.embed
!python src/run_extract_style_batch.py /content/list.txt --out /content/styles

In [ ]:
# 7. Послушать: ваш голос в TeraTTS против ближайшей заготовки
import onnxruntime as ort, unicodedata, math, json
from IPython.display import Audio, display
rep = json.load(open("/content/styles/extraction_report.json"))["styles"][0]
print(f"Заготовка-старт: {rep['preset']}, итоговая ошибка тембра {rep['loss']} (цель ≤ 0.24), шагов {rep['steps']}")
print(f"Темп речи: {rep['duration']['rate_ref']:.2f}/с в записи → {rep['duration']['rate_final']:.2f}/с у голоса")
S = lambda f: ort.InferenceSession(f"{MODELS}/{f}", providers=["CPUExecutionProvider"])
enc, dpm, smp, voc = S("text_encoder.onnx"), S("duration_predictor.onnx"), S("sampler_distilled_cfg3_8step.onnx"), S("vocoder.onnx")
table = json.load(open(f"{MODELS}/unicode_indexer.json"))
def tera(text, style_path, seed=1):
    st = json.load(open(style_path))
    ttl = np.asarray(st["style_ttl"]["data"], np.float32).reshape(1, 50, 256)
    dp = np.asarray(st["style_dp"]["data"], np.float32).reshape(1, 8, 16)
    t = unicodedata.normalize("NFKD", text)
    ids = np.array([[table[ord(c)] for c in t]], np.int64)
    dids = np.array([[table[ord(c)] for c in t.replace("+", "")]], np.int64)
    m = np.ones((1, 1, ids.shape[1]), np.float32)
    emb = enc.run(None, {"text_ids": ids, "style_ttl": ttl, "text_mask": m})[0]
    dur = float(dpm.run(None, {"text_ids": dids, "style_dp": dp, "text_mask": np.ones((1, 1, dids.shape[1]), np.float32)})[0][0]) / 1.05
    n = max(1, math.ceil(dur * 44100 / 3072))
    lat = smp.run(None, {"initial_latent": np.random.default_rng(seed).standard_normal((1, 144, n)).astype(np.float32),
                         "text_emb": emb, "style_ttl": ttl, "latent_mask": np.ones((1, 1, n), np.float32),
                         "text_mask": m, "guidance": np.asarray([3.0], np.float32)})[0]
    return voc.run(None, {"latent": lat})[0][0, :round(dur * 44100)]
TEXT = "<ru>Прив+ет! +Это м+ой г+олос. Тепер+ь +я м+огу сл+ушать кн+иги, кот+орые чит+аю с+ам.</ru>"
print("Ваша запись:"); display(Audio("/content/my_voice.wav"))
print("TeraTTS вашим голосом:"); display(Audio(tera(TEXT, "/content/styles/my_voice.json"), rate=44100))
print(f"Для сравнения — исходная заготовка {rep['preset']}:")
display(Audio(tera(TEXT, f"{PRESETS}/{rep['preset']}.json"), rate=44100))

In [ ]:
# 8. Скачать файл голоса
shutil.copy("/content/styles/my_voice.json", "/content/my_voice.json")
files.download("/content/my_voice.json")